<a href="https://colab.research.google.com/github/Krishnan-Raghavan/Packt/blob/main/DataCleaningAndPreparationChapter12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install required libraries

In [ ]:
!pip install transformers==4.42.4
!pip install beautifulsoup4==4.12.3
!pip install langchain-text-splitters==0.2.2
!pip install tiktoken==0.7.0
!pip install langchain==0.2.10
!pip install langchain-experimental==0.0.62
!pip install langchain-huggingface==0.0.3
!pip install presidio_analyzer==2.2.355
!pip install presidio_anonymizer==2.2.355
!pip install rapidfuzz-3.9.4 thefuzz-0.22.1
!pip install stanza==1.8.2
!pip install tf-keras-2.17.0

Text Cleaning

In [ ]:
from bs4 import BeautifulSoup
from transformers import BertTokenizer

# Sample user reviews
reviews = [
    "<html>This product    is <b>amazing!</b></html>",
    "The product is good, but it could be better!!!",
    "I've never seen such a terrible      product. 0/10",
    "The product is AWESOME!!! Highly recommended!",
]

# a. Removing HTML tags and Special Characters
def clean_html_tags(text):
    soup = BeautifulSoup(text, "html.parser")
    return soup.get_text()

# b. Handling Capitalization and Letter Case
def standardize_case(text):
    return text.lower()

# c. Dealing with Numerical Values and Symbols
def remove_numbers_and_symbols(text):
    return ''.join(e for e in text if e.isalpha() or e.isspace())

# d. Addressing Whitespace and Formatting Issues
def remove_extra_whitespace(text):
    return ' '.join(text.split())


# Applying the text preprocessing pipeline
def preprocess_text(text):
    text = clean_html_tags(text)
    text = standardize_case(text)
    text = remove_numbers_and_symbols(text)
    text = remove_extra_whitespace(text)
    return text

# Preprocess all reviews
preprocessed_reviews = [preprocess_text(review) for review in reviews]

print("Original Reviews:")
for review in reviews:
    print(f"- {review}")

print("\nPreprocessed Reviews:")
for preprocessed_review in preprocessed_reviews:
    print(f"- {preprocessed_review}")

Puntuation

In [ ]:
import string

# Sample text
text = "I love this product!!! It's amazing!!!"


# Option 1: Replace symbols and punctuation
replaced_text = text.translate(str.maketrans(string.punctuation, " " * len(string.punctuation)))
print("Replaced Text:", replaced_text)

# Option 2: Remove symbols and punctuation
removed_text = "".join(char for char in text if char.isalnum() or char.isspace())
print("Removed Text:", removed_text)

Personally Identifiable Information

In [ ]:
import pandas as pd
from presidio_analyzer import AnalyzerEngine
from presidio_anonymizer import AnonymizerEngine
from presidio_anonymizer.entities import OperatorConfig

# Sample DataFrame
data = {
    'text': [
        "Hello, my name is John Doe. My email is john.doe@example.com",
        "Contact Jane Smith at jane.smith@work.com",
        "Call her at 987-654-3210.",
        "This is a test message without PII."
    ]
}

df = pd.DataFrame(data)

# Initialize the analyzer and anonymizer engines
analyzer = AnalyzerEngine()
anonymizer = AnonymizerEngine()

def anonymize_text(text):
    """ Anonymize PII entities in text """
    # Analyze the text to detect PII entities
    analyzer_results = analyzer.analyze(text=text, entities=["PERSON", "EMAIL_ADDRESS", "PHONE_NUMBER"], language="en")

    # Define the anonymization configuration
    operators = {
        "PERSON": OperatorConfig("mask", {"masking_char": "*", "chars_to_mask": 4, "from_end": True}),
        "EMAIL_ADDRESS": OperatorConfig("mask", {"masking_char": "*", "chars_to_mask": 5, "from_end": True}),
        "PHONE_NUMBER": OperatorConfig("mask", {"masking_char": "*", "chars_to_mask": 6, "from_end": True})
    }

    # Anonymize the detected PII entities
    anonymized_result = anonymizer.anonymize(text=text, analyzer_results=analyzer_results, operators=operators)

    return anonymized_result.text

# Apply the anonymization function to the DataFrame
df['anonymized_text'] = df['text'].apply(anonymize_text)

# Display the DataFrame
print(df['anonymized_text'])

Dealing With Rare Words.

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

# Initialize the GPT-2 tokenizer and model
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")

# Define a text prompt with a rare word
text = "The quokka, a rare marsupial,"

# Encode the input text to tensor
indexed_tokens = tokenizer.encode(text, return_tensors='pt')

# Generate text until the output length reaches 50 tokens
output_text = model.generate(indexed_tokens, max_length=50, num_beams=5, no_repeat_ngram_size=2, early_stopping=True)

# Decode the output text
output_text_decoded = tokenizer.decode(output_text[0], skip_special_tokens=True)
print(output_text_decoded)

Spell Checker

In [ ]:
from transformers import pipeline

def fix_spelling(text):
    # Initialize the spelling correction pipeline
    spell_check = pipeline("text2text-generation", model="oliverguhr/spelling-correction-english-base")

    # Generate the corrected text
    corrected = spell_check(text, max_length=2048)[0]['generated_text']

    return corrected

# Test the function with some sample text containing spelling mistakes
sample_text = "y name si from Grece."
corrected_text = fix_spelling(sample_text)

print("Original text:", sample_text)
print("Corrected text:", corrected_text)

Fuzzy Matching

In [ ]:
!pip install thefuzz

In [ ]:
from transformers import pipeline
from thefuzz import process, fuzz

def fix_spelling(text, threshold=80):
    # Initialize the spelling correction pipeline
    spell_check = pipeline("text2text-generation", model="oliverguhr/spelling-correction-english-base")

    # Generate the corrected text
    corrected = spell_check(text, max_length=2048)[0]['generated_text']

    # Split the original and corrected texts into words
    original_words = text.split()
    corrected_words = corrected.split()

    # Create a dictionary of common English words (you can expand this list)
    common_words = set(['the', 'be', 'to', 'of', 'and', 'a', 'in', 'that', 'have', 'I', 'it', 'for', 'not', 'on', 'with', 'he', 'as', 'you', 'do', 'at'])

    # Fuzzy match each word
    final_words = []
    for orig, corr in zip(original_words, corrected_words):
        if orig.lower() in common_words:
            final_words.append(orig)  # Keep common words as they are
        else:
            # Use fuzzy matching to find the best match
            matches = process.extractOne(orig, [corr], scorer=fuzz.ratio)
            if matches[1] >= threshold:
                final_words.append(matches[0])
            else:
                final_words.append(orig)  # Keep the original word if no good match found

    return ' '.join(final_words)

# Test the function with some sample text containing spelling mistakes
sample_text = "Lets do a copmarsion of speling mistaks in this sentense."
corrected_text = fix_spelling(sample_text)

print("Original text:", sample_text)
print("Corrected text:", corrected_text)

Fixed Length Chunking

In [ ]:
# Step 1: Load Example Data
reviews = [
    "This smartphone has an excellent camera. The photos are sharp and the colors are vibrant. Overall, very satisfied with my purchase.",
    "I was disappointed with the laptop's performance. It frequently lags and the battery life is shorter than expected.",
    "The blender works great for making smoothies. It's powerful and easy to clean. Definitely worth the price.",
    "Customer support was unresponsive. I had to wait a long time for a reply, and my issue was not resolved satisfactorily.",
    "The book is a fascinating read. The storyline is engaging and the characters are well-developed. Highly recommend to all readers."
]

# Step 2: Create the TokenTextSplitter
from langchain_text_splitters import TokenTextSplitter

# Initialize the TokenTextSplitter with a chunk size of 50 tokens and no overlap
text_splitter = TokenTextSplitter(chunk_size=50, chunk_overlap=0)

# Step 3: Join Reviews and Split Text
# Combine the reviews into a single text block for chunking
text_block = " ".join(reviews)

# Split the text into token-based chunks
chunks = text_splitter.split_text(text_block)

# Print the chunks
print("Chunks with 50 tokens each:")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i + 1}:")
    print(chunk)
    print("\n")

# Step 4: Experiment with Different Chunk Sizes
chunk_sizes = [20, 70, 150]

for size in chunk_sizes:
    print(f"Chunk Size: {size}")
    text_splitter = TokenTextSplitter(chunk_size=size, chunk_overlap=0)
    chunks = text_splitter.split_text(text_block)

    for i, chunk in enumerate(chunks):
        print(f"Chunk {i + 1}:")
        print(chunk)
        print("\n")

Recursive Character Chunking

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

reviews = [
    "This smartphone has an excellent camera. The photos are sharp and the colors are vibrant. Overall, very satisfied with my purchase.",
    "I was disappointed with the laptop's performance. It frequently lags and the battery life is shorter than expected.",
    "The blender works great for making smoothies. It's powerful and easy to clean. Definitely worth the price.",
    "Customer support was unresponsive. I had to wait a long time for a reply, and my issue was not resolved satisfactorily.",
    "The book is a fascinating read. The storyline is engaging and the characters are well-developed. Highly recommend to all readers."
]

# Combine the reviews into a single text block for chunking
text_block = " ".join(reviews)

# Create a RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " ", ""],
    chunk_size=200,
    chunk_overlap=0,
    length_function=len
)

# Split the text into chunks
chunks = text_splitter.split_text(text_block)

# Print the chunks
for i, chunk in enumerate(chunks, 1):
    print(f"Chunk {i}:")
    print(chunk.strip())
    print("-" * 50)

Semantic Chunking

In [ ]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings
import os

reviews = [
    "This smartphone has an excellent camera. The photos are sharp and the colors are vibrant. Overall, very satisfied with my purchase.",
    "I was disappointed with the laptop's performance. It frequently lags and the battery life is shorter than expected.",
    "The blender works great for making smoothies. It's powerful and easy to clean. Definitely worth the price.",
    "Customer support was unresponsive. I had to wait a long time for a reply, and my issue was not resolved satisfactorily.",
    "The book is a fascinating read. The storyline is engaging and the characters are well-developed. Highly recommend to all readers."
]
# Combine the reviews into a single text block for chunking
text_block = " ".join(reviews)

text_splitter = SemanticChunker(HuggingFaceEmbeddings())

docs = text_splitter.create_documents([text_block])

for i, doc in enumerate(docs):
    print(f"Chunk {i + 1}:")
    print(doc.page_content)
    print("\n")

Word Tokenization

In [ ]:
import nltk
from nltk.tokenize import word_tokenize

# Download the necessary NLTK data (run this once)
nltk.download('punkt')

# Sample text
text = "The quick brown fox jumps over the lazy dog. It's unaffordable!"

# Perform word tokenization
word_tokens = word_tokenize(text)

print("Word tokens:")
print(word_tokens)

Byte Pair Encoding

In [ ]:
from tokenizers import Tokenizer

# Load the pre-trained GPT-2 BPE tokenizer
tokenizer = Tokenizer.from_pretrained("gpt2")

# Sample text
text = "Tokenization in medical texts can include words like hyperlipidemia.."

# Tokenize the text
encoding = tokenizer.encode(text)

# Print the tokens
print("Tokens:", encoding.tokens)

# Print the token IDs
print("Token IDs:", encoding.ids)

# Decode the token IDs back to text
decoded_text = tokenizer.decode(encoding.ids)
print("Decoded Text:", decoded_text)

Wordpiece Tokenization

In [ ]:
from transformers import BertTokenizer

# Load the pre-trained tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Sample text
text = "Tokenization in medical texts can include words like hyperlipidemia."


# Tokenize the text
tokens = tokenizer.tokenize(text)
print("Tokens:", tokens)

# Convert tokens to input IDs
input_ids = tokenizer.convert_tokens_to_ids(tokens)
print("Input IDs:", input_ids)

Specialised Tokenisers

In [ ]:
import stanza
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from collections import Counter
import numpy as np
import torch

# Initialize Stanza for biomedical text
stanza.download('en', package='mimic', processors='tokenize')
nlp = stanza.Pipeline('en', package='mimic', processors='tokenize')

# Initialize standard GPT-2 tokenizer
standard_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
standard_tokenizer.pad_token = standard_tokenizer.eos_token  # Set pad_token to eos_token
model = GPT2LMHeadModel.from_pretrained("gpt2")
model.config.pad_token_id = model.config.eos_token_id  # Set pad_token_id for the model

# Sample medical corpus
corpus = [
    "The patient suffered a myocardial infarction.",
    "Early detection of heart attack is crucial.",
    "Treatment for myocardial infarction includes medication.",
    "Patients with heart conditions require regular check-ups.",
    "Myocardial infarction can lead to severe complications."
]

def stanza_tokenize(text):
    doc = nlp(text)
    tokens = [word.text for sent in doc.sentences for word in sent.words]
    return tokens

def calculate_oov_and_compression(corpus, tokenizer):
    oov_count = 0
    total_tokens = 0
    all_tokens = []

    for sentence in corpus:
        tokens = tokenizer.tokenize(sentence) if hasattr(tokenizer, 'tokenize') else stanza_tokenize(sentence)
        all_tokens.extend(tokens)
        total_tokens += len(tokens)
        oov_count += tokens.count(tokenizer.oov_token) if hasattr(tokenizer, 'oov_token') else 0

    oov_rate = (oov_count / total_tokens) * 100 if total_tokens > 0 else 0
    avg_tokens_per_sentence = total_tokens / len(corpus)

    return oov_rate, avg_tokens_per_sentence, all_tokens

def analyze_token_utilization(tokens):
    token_counts = Counter(tokens)
    total_tokens = len(tokens)
    utilization = {token: count / total_tokens for token, count in token_counts.items()}
    return utilization

def calculate_perplexity(tokenizer, model, text):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        outputs = model(**inputs, labels=inputs["input_ids"])
    return torch.exp(outputs.loss).item()

# Evaluation
for tokenizer_name, tokenizer in [("Standard GPT-2", standard_tokenizer), ("Stanza Medical", stanza_tokenize)]:
    oov_rate, avg_tokens, all_tokens = calculate_oov_and_compression(corpus, tokenizer)
    utilization = analyze_token_utilization(all_tokens)

    print(f"\n{tokenizer_name} Tokenizer:")
    print(f"OOV Rate: {oov_rate:.2f}%")
    print(f"Average Tokens per Sentence: {avg_tokens:.2f}")
    print("Top 5 Most Used Tokens:")
    for token, freq in sorted(utilization.items(), key=lambda x: x[1], reverse=True)[:5]:
        print(f"  {token}: {freq:.2%}")


# Example output for "myocardial infarction"
term = "myocardial infarction"
print(f"\nTokenizing '{term}':")
print(f"Standard GPT-2: {standard_tokenizer.tokenize(term)}")
print(f"Stanza Medical: {stanza_tokenize(term)}")

Embedding BERT

In [ ]:
# Import necessary libraries
from transformers import BertTokenizer, BertModel
import torch

# Load pre-trained BERT tokenizer and model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

# Input sentence
sentence = "BERT embeddings are very useful for natural language processing tasks."

# Tokenize the input sentence
inputs = tokenizer(sentence, return_tensors='pt')

# Generate embeddings
with torch.no_grad():
    outputs = model(**inputs)

# Extract the last hidden states (embeddings)
last_hidden_states = outputs.last_hidden_state

# Print the shape of the embeddings tensor
print("Shape of the embeddings tensor:", last_hidden_states.shape)

# Print the embeddings for the first token (CLS token)
cls_embedding = last_hidden_states[0, 0, :].numpy()
print("CLS token embedding:", cls_embedding)

# Print the embeddings for the first word
first_word_embedding = last_hidden_states[0, 1, :].numpy()
print("First word embedding:", first_word_embedding)

BGE Embedding

In [ ]:
from langchain_community.embeddings import HuggingFaceBgeEmbeddings

# Define the model name and parameters
model_name = "BAAI/bge-small-en"
model_kwargs = {"device": "cpu"}
encode_kwargs = {"normalize_embeddings": True}

# Initialize the embeddings model
bge_embeddings = HuggingFaceBgeEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

# Sample sentences to embed
sentences = [
    "The quick brown fox jumps over the lazy dog.",
    "I love machine learning and natural language processing."
]

# Generate embeddings for each sentence
embeddings = [bge_embeddings.embed_query(sentence) for sentence in sentences]

# Print the embeddings
for i, embedding in enumerate(embeddings):
    print(f"Embedding for sentence {i+1}: {embedding[:5]}...")  # Print the first 5 values for brevity
    print(f"Length of embedding: {len(embedding)}")

General Text Embedding

In [ ]:
from sentence_transformers import SentenceTransformer

# Load the GTE-base model
model = SentenceTransformer('thenlper/gte-base')

# Sample texts to embed
texts = [
    "The quick brown fox jumps over the lazy dog.",
    "I love machine learning and natural language processing.",
    "Embeddings are useful for many NLP tasks."
]

# Generate embeddings
embeddings = model.encode(texts)

# Print the shape of the embeddings
print(f"Shape of embeddings: {embeddings.shape}")

# Print the first few values of the first embedding
print(f"First few values of the first embedding: {embeddings[0][:5]}")